# Teste — detetor de bola de padel (Roboflow)
Verifica se um modelo de bola de padel (treinado, YOLO, via Roboflow) **deteta a bola no teu vídeo** a 540p.
Corre poucas frames (barato na API gratuita). Se a bola aparecer, montamos o pipeline completo de tempo útil.

Precisas de uma conta **gratuita** em roboflow.com -> Settings -> API Key.

## 1. Instalar

In [ ]:
!pip install -q roboflow opencv-python-headless matplotlib
print('ok')

## 2. Montar Drive (onde está o vídeo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
VIDEO_PATH='/content/drive/MyDrive/PadelPro_Vision/videos/Analise Padel Modelo - Parada 4 min.mp4'
import os; print('video existe?', os.path.exists(VIDEO_PATH))

## 3. Carregar o modelo de bola
Cola a tua API key do Roboflow. O modelo por defeito e' o `rbada/padel-tennis-ball-detection` (v1).
Alternativa (descomenta): `padel-ll7pp/padel-dataset` (9156 imagens).

In [ ]:
API_KEY='COLA_AQUI_A_TUA_API_KEY'
from roboflow import Roboflow
rf=Roboflow(api_key=API_KEY)
project=rf.workspace('rbada').project('padel-tennis-ball-detection')
model=project.version(1).model
# ALT: project=rf.workspace('padel-ll7pp').project('padel-dataset'); model=project.version(1).model
print('modelo carregado')

## 4. Testar em ~24 frames espalhados pelo vídeo
Desenha um círculo onde o modelo deteta a bola. Confirma visualmente se acerta.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, tempfile, os
cap=cv2.VideoCapture(VIDEO_PATH); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps=cap.get(cv2.CAP_PROP_FPS)
idxs=[int(n*f) for f in np.linspace(0.18,0.95,24)]
tmp=tempfile.mkdtemp(); hits=0; imgs=[]
for i in idxs:
    cap.set(cv2.CAP_PROP_POS_FRAMES,i); ok,fr=cap.read()
    if not ok: continue
    p=os.path.join(tmp,'f.jpg'); cv2.imwrite(p,fr)
    try: preds=model.predict(p, confidence=12, overlap=30).json()['predictions']
    except Exception as e: preds=[]; print('erro pred:',e)
    if preds: hits+=1
    for d in preds:
        x,y,w,h=int(d['x']),int(d['y']),int(d['width']),int(d['height'])
        cv2.circle(fr,(x,y),max(8,w),(0,0,255),3)
        cv2.putText(fr,f"{d['confidence']:.2f}",(x+10,y),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,0,255),2)
    imgs.append(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
cap.release()
print(f'Bola detetada em {hits}/{len(imgs)} frames')
fig,axes=plt.subplots(4,6,figsize=(20,11))
for ax,im in zip(axes.ravel(),imgs): ax.imshow(im); ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Leitura
- Se a bola for detetada na maioria dos frames (circulos a acompanhar a bola real) -> **avancamos** para o pipeline completo (bola por frame -> deteccao_rallies -> corte do tempo util).
- Se quase nao detetar -> a 540p/angulo e' o limite, e a camara tem de melhorar.
Manda o print da grelha ao Claude.